# Privacy-Preserving Structural Perturbation (PPSP)
### Graph Laplacian + Frobenius-Regulated Adaptive Noise for IIoT Cloud Security
**Author:** Mahdi Mohammad Shibli · **Year:** 2026

---

## Verification Notes vs Research Specification

| Spec Claim | Verification Status | Notes |
|---|---|---|
| MI-based feature weighting | ✓ Implemented | `mutual_info_classif` on training labels |
| Frobenius norm regularisation | ✓ Implemented | `‖N‖_F ≤ δ_frob × ‖X‖_F` |
| Mirsky's theorem justification | ✓ Valid | Bounding ‖N‖_F bounds singular value shift |
| `target_epsilon_for_rose` missing in CFG | ✗ Fixed | Added to CFG |
| PPSPEngine uses y_labels from full dataset | ✗ Fixed | Now uses training labels only |
| SNR cell calls compute_snr with ppsp_adaptive | ✗ Fixed | ppsp_adaptive has no analytic SNR; skipped |
| Frobenius clipping breaks DP guarantee | ✗ Acknowledged | Treated as post-processing; empirical metric used |
| laplace_baseline clip wrong (clips output not noise) | ✗ Fixed | Noise truncated before addition |

## Pipeline

```
LOCAL                                           CLOUD
──────────────────────────────────────────      ─────────────────────
1. Load + group-stratified split      PRIVATE
2. Build KNN graph → smooth → PCA     PRIVATE
3. PPSP perturbation:                 PRIVATE
   a. MI weights from training data
   b. Weighted Laplace noise
   c. Frobenius norm clipping
4. Split Z_pert at n_train            PRIVATE
   ────────────────────────────────────────→   Z_train_pert (upload)
                                          →    Z_test_pert  (upload)
                                               Classifiers + adversary
```

| Cell | Purpose |
|------|---------|
| 0 | Install |
| 1 | Imports |
| 2 | Configuration |
| 3 | Data + group-stratified split |
| 4 | KNN graph + Laplacian smoothing + PCA |
| 5 | PPSP Engine (MI weights + Frobenius regularisation) |
| 6 | Classifier suite |
| 7 | Attribute inference attack (strongest adversary) |
| 8 | Full epsilon sweep — PPSP vs Laplace baseline |
| 9 | Visualisation |

## Cell 0 — Install

In [28]:
# !pip install networkx --quiet
# print("✓ Dependencies ready")

## Cell 1 — Imports

In [29]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.linalg import norm as matrix_norm
from scipy.sparse.csgraph import laplacian as sparse_laplacian
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph, KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✓ PPSP imports successful")

✓ PPSP imports successful


## Cell 2 — Configuration
> **Edit this cell only.**

In [30]:
CFG = {
    # ── Data ──────────────────────────────────────────────────────────────
    "data_path"               : "features_raw.csv",
    "label_column"            : "fault_type",
    "group_column"            : "file_id",
    "sensitive_class"         : 0,               # 0 = Ball fault
    "class_names"             : ["B", "IR", "Normal", "OR"],
    "test_size"               : 0.2,
    "random_state"            : 42,

    # ── Graph ─────────────────────────────────────────────────────────────
    "k_neighbors"             : 7,
    "alpha"                   : 0.7,             # Laplacian smoothing strength

    # ── PCA ───────────────────────────────────────────────────────────────
    "pca_variance"            : 0.90,
    "pca_max_components"      : 50,
    "target_epsilon_for_rose" : 1.0,             # FIX: was missing from spec CFG
    "target_snr"              : 2.0,             # Rose criterion (task-adapted)

    # ── PPSP (Privacy-Preserving Structural Perturbation) ─────────────────
    # delta_frob: Frobenius energy budget — noise ‖N‖_F ≤ delta_frob × ‖X‖_F
    # Mirsky's theorem: bounding ‖N‖_F ≤ δ bounds |σᵢ(X+N) - σᵢ(X)| ≤ ‖N‖_F
    # so singular values (spectral structure) of X are preserved.
    # Reference: Mirsky (1960); Tran et al. (2025)
    "delta_frob"              : 0.5,             # 50% energy budget

    # ── Sweep ─────────────────────────────────────────────────────────────
    "delta"                   : 1e-5,
    # "epsilons"                : [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0],
    "epsilons"                : [0.1, 1.0, 2.0],
    "methods"                 : ["ppsp_adaptive", "laplace_baseline"],

    # ── Evaluation ────────────────────────────────────────────────────────
    "plot_cms_sweep"          : False,
    "epsilon_highlight"       : 1.0,
}

print("✓ Configuration loaded")
print(f"  Sensitive class     : {CFG['sensitive_class']} ({CFG['class_names'][CFG['sensitive_class']]})")
print(f"  Frobenius budget    : {CFG['delta_frob']} (‖N‖_F ≤ {CFG['delta_frob']} × ‖X‖_F)")
print(f"  Methods             : {CFG['methods']}")
print(f"  Epsilon range       : {CFG['epsilons']}")

✓ Configuration loaded
  Sensitive class     : 0 (B)
  Frobenius budget    : 0.5 (‖N‖_F ≤ 0.5 × ‖X‖_F)
  Methods             : ['ppsp_adaptive', 'laplace_baseline']
  Epsilon range       : [0.1, 1.0, 2.0]


In [39]:
test     = pd.read_csv(CFG["data_path"])
test.shape

(17503, 131)

## Cell 3 — Data Loading + Group-Stratified Split
Split at **group level** (recording file) to prevent 50% window overlap leakage.

In [ ]:
df     = pd.read_csv(CFG["data_path"])
y      = LabelEncoder().fit_transform(df[CFG["label_column"]])
X      = df.drop(columns=[CFG["label_column"], CFG["group_column"]]).values
groups = df[CFG["group_column"]].values

group_ids    = np.unique(groups)
group_labels = np.array([int(pd.Series(y[groups == g]).mode()[0]) for g in group_ids])

print("Group-level class distribution:")
for cls, name in enumerate(CFG["class_names"]):
    print(f"  Class {cls} ({name:<8}) : {int((group_labels==cls).sum())} groups")

for cls in np.unique(group_labels):
    if int((group_labels==cls).sum()) < 2:
        raise ValueError(f"Class {CFG['class_names'][cls]} has < 2 groups.")

sss = StratifiedShuffleSplit(n_splits=1, test_size=CFG["test_size"],
                              random_state=CFG["random_state"])
tr_gi, te_gi = next(sss.split(group_ids, group_labels))
train_groups = set(group_ids[tr_gi])
test_groups  = set(group_ids[te_gi])
assert len(train_groups & test_groups) == 0, "Leakage!"

train_mask = np.isin(groups, list(train_groups))
test_mask  = np.isin(groups, list(test_groups))

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_all_sc = np.vstack([X_train_sc, X_test_sc])
y_all    = np.concatenate([y_train, y_test])
n_train  = len(X_train_sc)

print(f"\n✓ Split complete — no group overlap")
print(f"  Train : {X_train_sc.shape}  |  Test   ε=1.0             SVM    0.8123       0.8144   0.9697              0.7173        0.0957    0.500000: {X_test_sc.shape}  |  All : {X_all_sc.shape}")
for split_name, y_split in [("Train", y_train), ("Test", y_test)]:
    pcts = " | ".join(f"{n}:{(y_split==c).mean()*100:.1f}%" for c,n in enumerate(CFG["class_names"]))
    print(f"  {split_name:<8} {pcts}")

Group-level class distribution:
  Class 0 (B       ) : 16 groups
  Class 1 (IR      ) : 16 groups
  Class 2 (Normal  ) : 4 groups
  Class 3 (OR      ) : 28 groups

✓ Split complete — no group overlap
  Train : (13721, 129)  |  Test : (3782, 129)  |  All : (17503, 129)
  Train    B:22.4% | IR:22.4% | Normal:17.2% | OR:38.0%
  Test     B:18.7% | IR:18.7% | Normal:25.0% | OR:37.6%


## Cell 4 — KNN Graph + Laplacian Smoothing + PCA
Graph built on all N samples locally — never uploaded.
PCA fitted on training data only to avoid test leakage.

In [32]:
def build_sparse_knn_laplacian(X_scaled, k):
    knn   = kneighbors_graph(X_scaled, n_neighbors=k,
                              mode='distance', include_self=False, n_jobs=-1)
    dist  = knn.data
    sigma = float(np.median(dist)) or 1.0
    knn.data = np.exp(-(dist**2) / (2 * sigma**2))
    W = knn.maximum(knn.T).tocsr()
    L = sparse_laplacian(W, normed=False).tocsr()
    return W, L, sigma


def smooth_features_sparse(X, L_csr, alpha=0.5):
    """X_smooth = X - α · D⁻¹ · L · X  (sparse, no dense N×N)."""
    deg   = np.array(L_csr.diagonal(), dtype=np.float64)
    deg   = np.where(deg == 0, 1e-10, deg)
    D_inv = sp.diags(1.0 / deg)
    return X - alpha * D_inv.dot(L_csr).dot(X)


# ── Graph construction ────────────────────────────────────────────────────
N_all = X_all_sc.shape[0]
print(f"Building sparse KNN graph (N={N_all:,}, k={CFG['k_neighbors']})...")
W_all, L_all, sigma = build_sparse_knn_laplacian(X_all_sc, k=CFG["k_neighbors"])

dense_mb  = (N_all * N_all * 8) / 1e6
sparse_mb = (W_all.nnz * 8 * 3) / 1e6
row_sums  = np.array(L_all.sum(axis=1)).ravel()
print(f"  Non-zeros : {W_all.nnz:,}  |  {dense_mb:.0f} MB → {sparse_mb:.1f} MB  ({dense_mb/sparse_mb:.0f}× reduction)")
print(f"  Zero row-sum check: {np.allclose(row_sums, 0, atol=1e-6)}  (max dev: {np.max(np.abs(row_sums)):.2e})")

# ── Laplacian smoothing ───────────────────────────────────────────────────
print("Smoothing all features...")
X_all_smooth_clean   = smooth_features_sparse(X_all_sc, L_all, CFG["alpha"])
X_train_smooth_clean = X_all_smooth_clean[:n_train]
X_test_smooth_clean  = X_all_smooth_clean[n_train:]
print(f"  X_all_smooth_clean : {X_all_smooth_clean.shape}")

# ── PCA ───────────────────────────────────────────────────────────────────
pca_probe = PCA(n_components=min(CFG["pca_max_components"], X_all_smooth_clean.shape[1]))
pca_probe.fit(X_train_smooth_clean)
cum_var = np.cumsum(pca_probe.explained_variance_ratio_)
k_opt   = int(np.searchsorted(cum_var, CFG["pca_variance"]) + 1)
k_opt   = min(k_opt, CFG["pca_max_components"])
print(f"  PCA: {X_all_smooth_clean.shape[1]} → {k_opt} components ({cum_var[k_opt-1]*100:.1f}% variance)")

pca = PCA(n_components=k_opt, random_state=CFG["random_state"])
pca.fit(X_train_smooth_clean)

Z_all_smooth = pca.transform(X_all_smooth_clean)
Z_train      = Z_all_smooth[:n_train]
Z_test       = Z_all_smooth[n_train:]

# ── Std-proportional clip thresholds (for laplace_baseline comparison) ────
comp_stds   = Z_all_smooth.std(axis=0)
clip_factor = CFG["target_epsilon_for_rose"] / (2.0 * CFG["target_snr"] * np.sqrt(2))
clip_thresholds_pca = clip_factor * comp_stds
clip_thresholds_pca = np.where(clip_thresholds_pca == 0, 1e-10, clip_thresholds_pca)

print(f"  clip_factor={clip_factor:.6f}  (laplace_baseline reference)")
print(f"  Z_all_smooth : {Z_all_smooth.shape}")
print("\n✓ Cell 4 complete")

Building sparse KNN graph (N=17,503, k=7)...
  Non-zeros : 185,088  |  2451 MB → 4.4 MB  (552× reduction)
  Zero row-sum check: True  (max dev: 1.13e-14)
Smoothing all features...
  X_all_smooth_clean : (17503, 129)
  PCA: 129 → 31 components (90.4% variance)
  clip_factor=0.176777  (laplace_baseline reference)
  Z_all_smooth : (17503, 31)

✓ Cell 4 complete


## Cell 5 — PPSP Engine
### Privacy-Preserving Structural Perturbation

**Phase B — Feature Influence Scoring (MI-based)**

Mutual Information `I(Xⱼ; y)` measures how much feature j reduces uncertainty
about class labels. Higher MI = feature is more predictive = higher noise weight.
This concentrates noise budget on the features that leak the most class information.

**Phase C — Frobenius-Regulated Perturbation**

After generating weighted noise N, if `‖N‖_F > δ_frob × ‖X‖_F`, rescale N so
the total noise energy is bounded. By Mirsky's theorem:
`|σᵢ(X+N) - σᵢ(X)| ≤ ‖N‖_F ≤ δ_frob × ‖X‖_F`

This bounds how much the singular values (spectral structure) of the data
can shift, preserving the physical manifold while injecting privacy noise.

**Important caveat on formal DP:**
The Frobenius rescaling is a data-dependent post-processing step. It may
slightly modify the formal ε guarantee. This is treated as an empirical
privacy-utility mechanism. Privacy is measured via attribute inference attack,
not formal DP accounting.

**FIX from spec:** MI weights are computed from training labels only, not
from the full dataset. Using y_all would cause test set label leakage.

In [ ]:
class PPSPEngine:
    """
    Privacy-Preserving Structural Perturbation (PPSP) Engine.

    Combines MI-based adaptive noise allocation with Frobenius norm
    regularisation to bound the spectral distortion of the perturbed data.

    References:
        - Tran et al. (2025): Spectral Perturbation Bounds
        - Jiang et al. (2021): Adaptive DP for Industrial IoT
        - Mirsky (1960): Singular value perturbation bounds
    """

    def __init__(self, epsilon, delta_frob=0.1):
        self.epsilon    = epsilon
        self.delta_frob = delta_frob

    def compute_mi_weights(self, X_train, y_train):
        """
        Compute MI-based noise weights from TRAINING data only.
        Higher MI(Xⱼ; y) → feature j is more informative about class
        → assign more noise budget to feature j.

        Returns:
            weights : shape (d,), sums to 1.0
        """
        mi_scores = mutual_info_classif(
            X_train, y_train,
            discrete_features=False,
            random_state=CFG["random_state"],
        )
        weights = mi_scores / (np.sum(mi_scores) + 1e-10)
        return weights

    def perturb(self, X_all, y_train, n_train):
        """
        Apply PPSP to full feature matrix using training-derived MI weights.

        Args:
            X_all   : PCA-compressed smooth features, all samples (N × k)
            y_train : Training labels (n_train,) — used for MI only
            n_train : Training set size

        Returns:
            X_pert  : Perturbed features (N × k)
            info    : dict with perturbation diagnostics
        """
        N, d = X_all.shape

        # ── Phase B: MI weights from training data only ──────────────────
        weights = self.compute_mi_weights(X_all[:n_train], y_train)

        # ── Phase C-1: Generate weighted Laplace noise ───────────────────
        # scale_j = (weight_j × d) / ε
        # Features with high MI get higher noise scale
        noise_matrix = np.zeros((N, d), dtype=np.float64)
        noise_scales = np.zeros(d)
        for j in range(d):
            scale             = (weights[j] * d) / (self.epsilon + 1e-10)
            noise_matrix[:, j] = np.random.laplace(0, scale, N)
            noise_scales[j]    = scale

        # ── Phase C-2: Frobenius norm regularisation ─────────────────────
        # Bound: ‖N‖_F ≤ delta_frob × ‖X‖_F
        # If violated, rescale N proportionally (post-processing)
        original_frob     = matrix_norm(X_all, 'fro')
        noise_frob_before = matrix_norm(noise_matrix, 'fro')
        # Making the forb norm dependent on epsilon, so that smaller epsilon (stronger privacy) → smaller noise budget
        effective_frob    = self.delta_frob / np.sqrt(self.epsilon)
        max_allowed_noise = effective_frob * original_frob
        # max_allowed_noise = self.delta_frob * original_frob

        width = 45

        print(f"{'-'*60}")
        print(f"{'Matrix Perturbation & Privacy Budget':^60}")
        print(f"{'-'*60}")

        print(f"{'Original Frobenius Norm:':<{width}} {original_frob:>12.4f}")
        print(f"{'Initial Noise Frobenius Norm:':<{width}} {noise_frob_before:>12.4f}")
        print(f"{'Effective Budget Ratio (δ/sqrt(ε)):':<{width}} {effective_frob:>12.4f}")

        print(f"{'-'*60}")
        print(f"{'FINAL MAX ALLOWED NOISE:':<{width}} {max_allowed_noise:>12.4f}")
        print(f"{'-'*60}")

        rescaled = False
        if noise_frob_before > max_allowed_noise:
            scale_factor  = max_allowed_noise / noise_frob_before
            noise_matrix  = noise_matrix * scale_factor
            rescaled      = True

        noise_frob_after = matrix_norm(noise_matrix, 'fro')

        return X_all + noise_matrix, {
            "method"            : "ppsp_adaptive",
            "epsilon"           : self.epsilon,
            "delta_frob"        : self.delta_frob,
            "original_frob"     : round(float(original_frob), 4),
            "noise_frob_before" : round(float(noise_frob_before), 4),
            "noise_frob_after"  : round(float(noise_frob_after), 4),
            "max_allowed"       : round(float(max_allowed_noise), 4),
            "was_rescaled"      : rescaled,
            "frob_ratio"        : round(float(noise_frob_after / original_frob), 6),
            "top_mi_features"   : np.argsort(weights)[-3:][::-1].tolist(),
            "noise_scales"      : noise_scales,
        }


def perturb_smooth_features(X_smooth, epsilon, method="ppsp_adaptive",
                              y_labels=None, n_train=None,
                              clip_thresholds=None, delta=1e-5):
    """
    Unified perturbation switch for the epsilon sweep.

    Args:
        X_smooth      : PCA-compressed smooth features (N × k)
        epsilon       : Privacy budget
        method        : ppsp_adaptive | laplace_baseline
        y_labels      : Training labels (required for ppsp_adaptive)
        n_train       : Training set size (required for ppsp_adaptive)
        clip_thresholds: Per-component thresholds (required for laplace_baseline)
        delta         : DP delta (laplace_baseline only)
    """
    if method == "ppsp_adaptive":
        if y_labels is None or n_train is None:
            raise ValueError("ppsp_adaptive requires y_labels and n_train.")
        engine = PPSPEngine(epsilon=epsilon, delta_frob=CFG["delta_frob"])
        X_pert, info = engine.perturb(X_smooth, y_labels, n_train)
        return X_pert, info

    elif method == "laplace_baseline":
        if clip_thresholds is None:
            raise ValueError("laplace_baseline requires clip_thresholds.")
        # Truncated Laplace — same as proven pipeline
        X_clipped    = np.clip(X_smooth, -clip_thresholds, clip_thresholds)
        delta_f      = 2.0 * clip_thresholds
        noise_scales = delta_f / epsilon
        noise        = np.random.laplace(0, 1, size=X_smooth.shape) * noise_scales
        noise        = np.clip(noise, -clip_thresholds, clip_thresholds)
        return X_clipped + noise, {
            "method"          : "laplace_baseline",
            "noise_scales"    : noise_scales,
            "rep_noise_scale" : float(np.median(noise_scales)),
        }

    else:
        raise ValueError(f"Unknown method '{method}'. Choose: ppsp_adaptive | laplace_baseline")


# ── Diagnostic: verify PPSP at ε=1.0 ─────────────────────────────────────
print("PPSP Engine verification at ε=1.0:")
engine_test   = PPSPEngine(epsilon=1.0, delta_frob=CFG["delta_frob"])
_, info_test  = engine_test.perturb(Z_all_smooth, y_train, n_train)

print(f"  Original ‖X‖_F       : {info_test['original_frob']}")
print(f"  Noise ‖N‖_F (before) : {info_test['noise_frob_before']}")
print(f"  Max allowed noise    : {info_test['max_allowed']}")
print(f"  Noise ‖N‖_F (after)  : {info_test['noise_frob_after']}")
print(f"  Was rescaled?        : {info_test['was_rescaled']}")
print(f"  Frob ratio           : {info_test['frob_ratio']:.6f}  (= noise/signal, should ≤ {CFG['delta_frob']})")
print(f"  Top MI components    : {info_test['top_mi_features']}  (most noise budget here)")
print("\n✓ Cell 5 complete — PPSP Engine ready")

PPSP Engine verification at ε=1.0:
  Original ‖X‖_F       : 1285.4202
  Noise ‖N‖_F (before) : 1615.5914
  Max allowed noise    : 642.7101
  Noise ‖N‖_F (after)  : 642.7101
  Was rescaled?        : True
  Frob ratio           : 0.500000  (= noise/signal, should ≤ 0.5)
  Top MI components    : [0, 2, 1]  (most noise budget here)

✓ Cell 5 complete — PPSP Engine ready


## Cell 6 — Classifier Suite

In [34]:
try:
    from cuml.ensemble     import RandomForestClassifier  as cuRF
    from cuml.ensemble     import GradientBoostingClassifier as cuGB
    from cuml.svm          import SVC                     as cuSVC
    from cuml.neighbors    import KNeighborsClassifier    as cuKNN
    from cuml.linear_model import LogisticRegression      as cuLR
    USE_GPU = True
    print("✓ cuML — GPU classifiers")
except ImportError:
    USE_GPU = False
    print("⚠  CPU classifiers")

if USE_GPU:
    CLASSIFIERS = {
        "Logistic Regression" : cuLR(max_iter=500),
        "Random Forest"       : cuRF(n_estimators=100, random_state=CFG["random_state"]),
        "SVM"                 : cuSVC(kernel="rbf", probability=True),
        "KNN"                 : cuKNN(n_neighbors=5),
        "Gradient Boosting"   : cuGB(n_estimators=100, random_state=CFG["random_state"]),
    }
else:
    CLASSIFIERS = {
        "Logistic Regression" : LogisticRegression(max_iter=500, solver="saga", random_state=CFG["random_state"]),
        "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=CFG["random_state"]),
        "SVM"                 : SVC(kernel="rbf", probability=True, random_state=CFG["random_state"]),
        "KNN"                 : KNeighborsClassifier(n_neighbors=5),
        "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=CFG["random_state"]),
    }


def evaluate_classifier(clf, X_te, y_te):
    y_pred = clf.predict(X_te)
    if hasattr(y_pred, "to_numpy"): y_pred = y_pred.to_numpy()
    n_cls  = len(np.unique(y_te))
    try:
        proba = clf.predict_proba(X_te)
        if hasattr(proba, "to_numpy"): proba = proba.to_numpy()
        auc   = (roc_auc_score(y_te, proba[:,1]) if n_cls == 2
                 else roc_auc_score(y_te, proba, multi_class="ovr", average="macro"))
    except Exception:
        auc = np.nan
    return {
        "accuracy"    : round(accuracy_score(y_te, y_pred), 4),
        "f1_macro"    : round(f1_score(y_te, y_pred, average="macro",    zero_division=0), 4),
        "f1_weighted" : round(f1_score(y_te, y_pred, average="weighted", zero_division=0), 4),
        "precision"   : round(precision_score(y_te, y_pred, average="weighted", zero_division=0), 4),
        "recall"      : round(recall_score(y_te, y_pred, average="weighted",    zero_division=0), 4),
        "roc_auc"     : round(float(auc), 4) if not np.isnan(auc) else np.nan,
        "conf_matrix" : confusion_matrix(y_te, y_pred),
    }


def run_classifiers(X_tr, X_te, y_tr, y_te, label="", class_names=None, verbose=True):
    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)
    results = {}
    if verbose and label:
        print(f"\n  {'Classifier':<25} {'Acc':>6} {'F1-Mac':>7} "
              f"{'F1-Wt':>7} {'Prec':>6} {'Rec':>6} {'AUC':>7}")
        print(f"  {'─'*25} {'─'*6} {'─'*7} {'─'*7} {'─'*6} {'─'*6} {'─'*7}")
    for name, clf in CLASSIFIERS.items():
        clf.fit(X_tr_sc, y_tr)
        m = evaluate_classifier(clf, X_te_sc, y_te)
        results[name] = m
        if verbose:
            auc_s = f"{m['roc_auc']:>7.4f}" if not np.isnan(m["roc_auc"]) else "   N/A "
            print(f"  {name:<25} {m['accuracy']:>6.4f} {m['f1_macro']:>7.4f} "
                  f"{m['f1_weighted']:>7.4f} {m['precision']:>6.4f} "
                  f"{m['recall']:>6.4f} {auc_s}")
    return results


def results_to_df(results):
    rows = [{"classifier": k, **{m: v for m, v in met.items() if m != "conf_matrix"}}
            for k, met in results.items()]
    df = pd.DataFrame(rows).set_index("classifier")
    return df[["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]]


print("=" * 70)
print("BASELINE — clean PCA-smoothed features (no perturbation)")
print("=" * 70)
baseline_results = run_classifiers(Z_train, Z_test, y_train, y_test,
                                    label="Baseline", class_names=CFG["class_names"])
baseline_df = results_to_df(baseline_results)
print(f"\n  Best by F1-weighted : {baseline_df['f1_weighted'].idxmax()}  "
      f"({baseline_df['f1_weighted'].max():.4f})")

⚠  CPU classifiers
BASELINE — clean PCA-smoothed features (no perturbation)

  Classifier                   Acc  F1-Mac   F1-Wt   Prec    Rec     AUC
  ───────────────────────── ────── ─────── ─────── ────── ────── ───────
  Logistic Regression       0.8173  0.8138  0.8134 0.8170 0.8173  0.9243
  Random Forest             0.9268  0.9175  0.9228 0.9330 0.9268  0.9961
  SVM                       0.9601  0.9570  0.9590 0.9620 0.9601  0.9973
  KNN                       0.8612  0.8630  0.8676 0.9070 0.8612  0.9526
  Gradient Boosting         0.9273  0.9150  0.9234 0.9329 0.9273  0.9954

  Best by F1-weighted : SVM  (0.9590)


## Cell 7 — Attribute Inference Attack
Uses worst-case adversary (LR + RF + SVM), reports highest accuracy.

In [35]:
def run_inference_attack(X_tr, X_te, y_tr, y_te, sensitive_class,
                         return_all=False):
    """
    Attribute Inference Attack — worst-case adversary.

    Trains LR, RF, SVM; reports highest adversary accuracy.
    Reference: Jayaraman & Evans (2022).
    """
    y_s_tr = (y_tr == sensitive_class).astype(int)
    y_s_te = (y_te == sensitive_class).astype(int)

    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    adversaries = {
        "LR"  : LogisticRegression(max_iter=500, solver="saga",
                                    class_weight="balanced", random_state=42),
        # "RF"  : RandomForestClassifier(n_estimators=100,
        #                                 class_weight="balanced", random_state=42),
        # "SVM" : SVC(kernel="rbf", probability=False,
        #             class_weight="balanced", random_state=42),
    }

    rand_bl      = float(max(y_s_te.mean(), 1.0 - y_s_te.mean()))
    results_each = {}
    worst_acc    = 0.0
    worst_name   = "LR"

    for name, clf in adversaries.items():
        clf.fit(X_tr_sc, y_s_tr)
        acc = accuracy_score(y_s_te, clf.predict(X_te_sc))
        results_each[name] = round(acc, 4)
        if acc > worst_acc:
            worst_acc  = acc
            worst_name = name

    out = {
        "adversary_accuracy" : round(worst_acc, 4),
        "worst_adversary"    : worst_name,
        "random_baseline"    : round(rand_bl, 4),
        "privacy_gain"       : round(rand_bl - worst_acc, 4),
    }
    if return_all:
        out["per_adversary"] = results_each
    return out


print("Adversary sanity checks (no perturbation):")
print(f"  {'Label':<40} {'Worst adv':>10} {'Model':>6} {'Baseline':>10} {'Gain':>8}  {'LR':>7} {'RF':>7} {'SVM':>7}")
print(f"  {'─'*40} {'─'*10} {'─'*6} {'─'*10} {'─'*8}  {'─'*7} {'─'*7} {'─'*7}")
for label, Xtr, Xte in [
    ("Raw scaled features",         X_train_sc, X_test_sc),
    ("Clean PCA-smoothed (Z_train)", Z_train,    Z_test),
]:
    p  = run_inference_attack(Xtr, Xte, y_train, y_test,
                               CFG["sensitive_class"], return_all=True)
    ea = p["per_adversary"]
    s  = "✓ protecting" if p["privacy_gain"] > 0 else "✗ leaking"
    print(f"  {label:<40} {p['adversary_accuracy']:>10.4f} "
          f"{p['worst_adversary']:>6} "
          f"{p['random_baseline']:>10.4f} "
          f"{p['privacy_gain']:>+8.4f}  "
        #   f"{ea['LR']:>7.4f} {ea['RF']:>7.4f} {ea['SVM']:>7.4f}  {s}"
    )

Adversary sanity checks (no perturbation):
  Label                                     Worst adv  Model   Baseline     Gain       LR      RF     SVM
  ──────────────────────────────────────── ────────── ────── ────────── ────────  ─────── ─────── ───────
  Raw scaled features                          0.9804     LR     0.8131  -0.1674  
  Clean PCA-smoothed (Z_train)                 0.9162     LR     0.8131  -0.1031  


## Cell 8 — Full Epsilon Sweep
Compares PPSP (adaptive MI + Frobenius) against truncated Laplace baseline
across all epsilon values. Both operate on the same `Z_all_smooth` input.

In [36]:
import time

def run_epsilon_sweep(
    Z_all_smooth, y_train, y_test, n_train,
    method="ppsp_adaptive", sensitive_class=0,
    epsilons=None, clip_thresholds=None,
    class_names=None, plot_cms_sweep=False,
    baseline_results=None,
):
    """
    Full epsilon sweep for PPSP or laplace_baseline.

    For each ε:
        1. Perturb Z_all_smooth → Z_all_pert   (LOCAL)
        2. Split at n_train                     (LOCAL)
        3. Classify on Z_pert                   (simulates CLOUD)
        4. Adversary attack on Z_pert           (privacy evaluation)
    """
    if epsilons is None: epsilons = CFG["epsilons"]

    records       = []
    sweep_records = []

    # ── Baseline ──────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print(f"  SWEEP — {method.upper()}")
    print("="*70)

    if baseline_results is None:
        print("  Computing baseline...")
        base_clf = run_classifiers(Z_all_smooth[:n_train], Z_all_smooth[n_train:],
                                    y_train, y_test, label="Baseline",
                                    class_names=class_names)
        base_prv = run_inference_attack(Z_all_smooth[:n_train], Z_all_smooth[n_train:],
                                         y_train, y_test, sensitive_class)
    else:
        base_clf, base_prv = baseline_results
        print("  (Baseline reused)")

    base_df = results_to_df(base_clf)
    best_b  = base_df["f1_weighted"].idxmax()
    records.append({
        "epsilon": np.nan, "label": "Baseline", "best_classifier": best_b,
        **{k: base_df.loc[best_b, k]
           for k in ["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]},
        **base_prv,
        "frob_ratio": 0.0, "was_rescaled": False,
    })

    # ── Epsilon loop ──────────────────────────────────────────────────────
    for i, eps in enumerate(epsilons):
        t0 = time.time()
        print(f"\n{'─'*70}")
        print(f"  ε = {eps:<8}  |  {method}  [{i+1}/{len(epsilons)}]")
        print("─"*70)

        if method == "ppsp_adaptive":
            Z_all_pert, pinfo = perturb_smooth_features(
                Z_all_smooth, eps, method="ppsp_adaptive",
                y_labels=y_train, n_train=n_train,
            )
            rescaled_str = "rescaled ✓" if pinfo["was_rescaled"] else "no rescale"
            print(f"  frob_ratio={pinfo['frob_ratio']:.6f}  "
                  f"noise_frob={pinfo['noise_frob_after']:.4f}  "
                  f"max_allowed={pinfo['max_allowed']:.4f}  {rescaled_str}")
            extra = {"frob_ratio": pinfo["frob_ratio"],
                     "was_rescaled": pinfo["was_rescaled"]}

        elif method == "laplace_baseline":
            if clip_thresholds is None:
                raise ValueError("laplace_baseline requires clip_thresholds.")
            Z_all_pert, pinfo = perturb_smooth_features(
                Z_all_smooth, eps, method="laplace_baseline",
                clip_thresholds=clip_thresholds,
            )
            print(f"  noise_scale={pinfo['rep_noise_scale']:.6f}")
            extra = {"frob_ratio": float(matrix_norm(Z_all_pert - Z_all_smooth, 'fro') /
                                         matrix_norm(Z_all_smooth, 'fro')),
                     "was_rescaled": False}
        else:
            raise ValueError(f"Unknown method: {method}")

        Z_tr_pert = Z_all_pert[:n_train]
        Z_te_pert = Z_all_pert[n_train:]

        clf_res = run_classifiers(Z_tr_pert, Z_te_pert, y_train, y_test,
                                   label=f"ε={eps}", class_names=class_names)
        clf_df  = results_to_df(clf_res)
        best_c  = clf_df["f1_weighted"].idxmax()

        prv = run_inference_attack(Z_tr_pert, Z_te_pert, y_train, y_test,
                                    sensitive_class)
        elapsed = time.time() - t0
        print(f"\n  Best: {best_c:<25}  adv={prv['adversary_accuracy']:.4f}  "
              f"baseline={prv['random_baseline']:.4f}  "
              f"gain={prv['privacy_gain']:+.4f}  "
              f"({'✓' if prv['privacy_gain']>0 else '✗'})  [{elapsed:.1f}s]")

        if plot_cms_sweep:
            plot_confusion_matrices(clf_res, class_names, label=f"{method} ε={eps}")

        row = {
            "epsilon"        : eps,
            "label"          : f"ε={eps}",
            "best_classifier": best_c,
            **{k: clf_df.loc[best_c, k]
               for k in ["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]},
            **prv, **extra,
        }
        records.append(row)
        sweep_records.append(row)

    df = pd.DataFrame(records)
    print("\n\n" + "="*70)
    print(f"  RESULTS — {method}")
    print("="*70)
    print(df[["label","best_classifier","accuracy","f1_weighted",
              "roc_auc","adversary_accuracy","privacy_gain","frob_ratio"]]
          .to_string(index=False))
    return df, sweep_records


# ── Compute baseline once ─────────────────────────────────────────────────
print("Computing baseline (runs once, shared across methods)...")
t0 = time.time()
_base_clf = run_classifiers(Z_train, Z_test, y_train, y_test,
                             label="Baseline", class_names=CFG["class_names"])
_base_prv = run_inference_attack(Z_train, Z_test, y_train, y_test,
                                  CFG["sensitive_class"])
print(f"  Baseline done in {time.time()-t0:.1f}s")

# ── Run sweep for all methods ─────────────────────────────────────────────
sweep_results = {}
sweep_records = {}

for method in CFG["methods"]:
    df, records = run_epsilon_sweep(
        Z_all_smooth    = Z_all_smooth,
        y_train         = y_train,
        y_test          = y_test,
        n_train         = n_train,
        method          = method,
        sensitive_class = CFG["sensitive_class"],
        epsilons        = CFG["epsilons"],
        clip_thresholds = clip_thresholds_pca,
        class_names     = CFG["class_names"],
        plot_cms_sweep  = CFG["plot_cms_sweep"],
        baseline_results= (_base_clf, _base_prv),
    )
    sweep_results[method] = df
    sweep_records[method] = records

Computing baseline (runs once, shared across methods)...

  Classifier                   Acc  F1-Mac   F1-Wt   Prec    Rec     AUC
  ───────────────────────── ────── ─────── ─────── ────── ────── ───────
  Logistic Regression       0.8173  0.8138  0.8134 0.8170 0.8173  0.9243
  Random Forest             0.9268  0.9175  0.9228 0.9330 0.9268  0.9961
  SVM                       0.9601  0.9570  0.9590 0.9620 0.9601  0.9973
  KNN                       0.8612  0.8630  0.8676 0.9070 0.8612  0.9526
  Gradient Boosting         0.9273  0.9150  0.9234 0.9329 0.9273  0.9954
  Baseline done in 95.2s

  SWEEP — PPSP_ADAPTIVE
  (Baseline reused)

──────────────────────────────────────────────────────────────────────
  ε = 0.1       |  ppsp_adaptive  [1/3]
──────────────────────────────────────────────────────────────────────
  frob_ratio=1.581139  noise_frob=2032.4279  max_allowed=2032.4279  rescaled ✓

  Classifier                   Acc  F1-Mac   F1-Wt   Prec    Rec     AUC
  ───────────────────────

## Cell 9 — Visualisation

In [37]:
# def plot_confusion_matrices(results, class_names=None, label=""):
#     n = len(results); ncols = min(3,n); nrows = int(np.ceil(n/ncols))
#     fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 4.5*nrows))
#     axes = np.array(axes).flatten()
#     for idx, (name, m) in enumerate(results.items()):
#         ConfusionMatrixDisplay(m["conf_matrix"], display_labels=class_names).plot(
#             ax=axes[idx], colorbar=False, cmap="Blues")
#         axes[idx].set_title(f"{name}\nAcc={m['accuracy']:.3f}  F1={m['f1_weighted']:.3f}", fontsize=10)
#         axes[idx].tick_params(axis="x", rotation=45)
#     for idx in range(len(results), len(axes)): axes[idx].set_visible(False)
#     plt.suptitle(f"Confusion Matrices — {label}", fontsize=13, y=1.01)
#     plt.tight_layout()
#     fname = f"confusion_{label.replace(' ','_').replace('=','')}.png"
#     plt.savefig(fname, dpi=200, bbox_inches="tight"); plt.show()
#     print(f"  Saved: {fname}")


# def plot_ppsp_comparison(sweep_results):
#     """Privacy-utility tradeoff: PPSP vs Laplace baseline."""
#     method_styles = {
#         "ppsp_adaptive"   : {"color": "tab:blue",   "label": "PPSP (MI + Frobenius)", "marker": "o"},
#         "laplace_baseline": {"color": "tab:orange",  "label": "Laplace Baseline",       "marker": "s"},
#     }

#     fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#     # Left — Utility
#     ax = axes[0]
#     for method, df in sweep_results.items():
#         sweep = df[df["epsilon"].notna()].copy()
#         style = method_styles.get(method, {"color":"grey","label":method,"marker":"o"})
#         ax.plot(sweep["epsilon"], sweep["f1_weighted"],
#                 color=style["color"], marker=style["marker"],
#                 linewidth=2.5, markersize=7, label=style["label"])
#     base_f1 = list(sweep_results.values())[0]
#     base_f1 = base_f1[base_f1["label"]=="Baseline"]["f1_weighted"].values[0]
#     ax.axhline(base_f1, color="grey", linestyle=":", linewidth=1.2,
#                label=f"Baseline F1 ({base_f1:.3f})")
#     ax.set_xscale("log")
#     ax.set_xlabel("ε (log scale)\n← more private    less private →", fontsize=11)
#     ax.set_ylabel("F1-Weighted (Utility)", fontsize=11)
#     ax.set_ylim(0, 1.05); ax.legend(fontsize=9)
#     ax.grid(True, which="both", linestyle="--", alpha=0.4)
#     ax.set_title("Utility vs ε", fontsize=12)

#     # Right — Privacy
#     ax = axes[1]
#     for method, df in sweep_results.items():
#         sweep = df[df["epsilon"].notna()].copy()
#         style = method_styles.get(method, {"color":"grey","label":method,"marker":"o"})
#         ax.plot(sweep["epsilon"], sweep["adversary_accuracy"],
#                 color=style["color"], marker=style["marker"],
#                 linewidth=2.5, markersize=7, linestyle="--", label=style["label"])
#     rand_bl = list(sweep_results.values())[0]
#     rand_bl = rand_bl[rand_bl["label"]=="Baseline"]["random_baseline"].values[0]
#     ax.axhline(rand_bl, color="black", linestyle=":", linewidth=1.5,
#                label=f"Random baseline ({rand_bl:.3f})")
#     ax.set_xscale("log")
#     ax.set_xlabel("ε (log scale)\n← more private    less private →", fontsize=11)
#     ax.set_ylabel("Adversary Accuracy (↓ = better privacy)", fontsize=11)
#     ax.set_ylim(0, 1.05); ax.legend(fontsize=9)
#     ax.grid(True, which="both", linestyle="--", alpha=0.4)
#     ax.set_title("Privacy (Adversary) vs ε\nBelow dashed = genuine protection", fontsize=12)

#     plt.suptitle("PPSP vs Laplace Baseline — Privacy-Utility Tradeoff\n"
#                  "Graph-smoothed + MI-weighted + Frobenius-regulated perturbation",
#                  fontsize=13)
#     plt.tight_layout()
#     plt.savefig("ppsp_comparison.png", dpi=300, bbox_inches="tight")
#     plt.savefig("ppsp_comparison.pdf", bbox_inches="tight")
#     plt.show()
#     print("  Saved: ppsp_comparison.png")


# def plot_frob_analysis(sweep_results):
#     """Show how Frobenius ratio evolves with epsilon."""
#     fig, ax = plt.subplots(figsize=(10, 5))
#     for method, df in sweep_results.items():
#         sweep = df[df["epsilon"].notna()].copy()
#         label = "PPSP" if method == "ppsp_adaptive" else "Laplace"
#         ax.plot(sweep["epsilon"], sweep["frob_ratio"],
#                 marker="o", linewidth=2, label=f"{label} — ‖N‖_F / ‖X‖_F")
#     ax.axhline(CFG["delta_frob"], color="red", linestyle="--", linewidth=1.5,
#                label=f"Frobenius budget δ={CFG['delta_frob']}")
#     ax.set_xscale("log")
#     ax.set_xlabel("ε", fontsize=11)
#     ax.set_ylabel("‖N‖_F / ‖X‖_F  (noise energy / signal energy)", fontsize=11)
#     ax.legend(fontsize=9)
#     ax.grid(True, which="both", linestyle="--", alpha=0.4)
#     ax.set_title("Frobenius Noise Ratio vs ε\n"
#                  "PPSP enforces ≤ δ_frob=0.1 by rescaling; Laplace may exceed it",
#                  fontsize=12)
#     plt.tight_layout()
#     plt.savefig("frobenius_analysis.png", dpi=200, bbox_inches="tight")
#     plt.show()
#     print("  Saved: frobenius_analysis.png")


# def plot_privacy_gain_comparison(sweep_results):
#     """Direct privacy gain comparison — positive = adversary below random."""
#     fig, ax = plt.subplots(figsize=(10, 5))
#     method_styles = {
#         "ppsp_adaptive"   : {"color": "tab:blue",  "label": "PPSP Adaptive"},
#         "laplace_baseline": {"color": "tab:orange", "label": "Laplace Baseline"},
#     }
#     for method, df in sweep_results.items():
#         sweep = df[df["epsilon"].notna()].copy()
#         style = method_styles.get(method, {"color": "grey", "label": method})
#         ax.plot(sweep["epsilon"], sweep["privacy_gain"],
#                 color=style["color"], marker="o", linewidth=2.5,
#                 markersize=7, label=style["label"])
#     ax.axhline(0, color="black", linestyle="-", linewidth=1.5, alpha=0.7,
#                label="Zero gain (adversary = random baseline)")
#     ax.fill_between(sweep["epsilon"], 0,
#                     ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 0.1,
#                     alpha=0.05, color="green", label="Privacy protected zone")
#     ax.set_xscale("log")
#     ax.set_xlabel("ε", fontsize=11)
#     ax.set_ylabel("Privacy Gain = random_baseline − adversary_accuracy", fontsize=11)
#     ax.legend(fontsize=9)
#     ax.grid(True, which="both", linestyle="--", alpha=0.4)
#     ax.set_title("Privacy Gain vs ε\nPositive = adversary worse than random guessing", fontsize=12)
#     plt.tight_layout()
#     plt.savefig("privacy_gain_comparison.png", dpi=200, bbox_inches="tight")
#     plt.show()
#     print("  Saved: privacy_gain_comparison.png")


# # ── Generate all plots ────────────────────────────────────────────────────
# plot_ppsp_comparison(sweep_results)
# plot_frob_analysis(sweep_results)
# plot_privacy_gain_comparison(sweep_results)

# # Confusion matrix at epsilon_highlight
# eps_h = CFG["epsilon_highlight"]
# for method in CFG["methods"]:
#     print(f"\n  Confusion matrix — {method} at ε={eps_h}")
#     if method == "ppsp_adaptive":
#         Z_h, _ = perturb_smooth_features(Z_all_smooth, eps_h,
#                                           method="ppsp_adaptive",
#                                           y_labels=y_train, n_train=n_train)
#     else:
#         Z_h, _ = perturb_smooth_features(Z_all_smooth, eps_h,
#                                           method="laplace_baseline",
#                                           clip_thresholds=clip_thresholds_pca)
#     res_h = run_classifiers(Z_h[:n_train], Z_h[n_train:], y_train, y_test,
#                              class_names=CFG["class_names"], verbose=False)
#     plot_confusion_matrices(res_h, CFG["class_names"], label=f"{method} ε={eps_h}")

# print("\n✓ All plots saved.")